# Colocação de Paineis

Pretende-se decorar uma parede com uma sequência de 7 painéis, de iguais dimensões, justapostos. Os painéis podem ser de: (1) madeira, (2) metal ou (3) vidro. A colocação dos painéis tem que ser feita de acordo com as seguintes regras:
  1. Cada uma das 7 posições possíveis tem que ter um painel.
  2. Os painéis não podem ser sobrepostos.
  3. Os painéis das extremidades não podem ser do mesmo tipo de material.
  4. O painel do meio não pode ser de vidro.
  5. Todos os materiais têm que ser usados.
  6. Não podem ficar dois painéis de vidro seguidos.
  7. Junto a um painel de metal terá de haver pelo menos um painel de madeira.
  8. Se o primeiro painel for de vidro e o segundo de madeira, o último terá que ser de metal.

Será que este conjunto de regras é consistente? Isto é, será que este problema tem solução? Apresente uma solução, caso exista.

Comece por instalar o Z3.
Tem [aqui](https://ericpony.github.io/z3py-tutorial/guide-examples.htm) um pequeno tutorial do Z3Py.

In [1]:
!pip install z3-solver

Para codificar este problema em lógica proposicional, temos que

1. Declarar um conjunto de variáveis Boolenas
$x_{p,m}$, para $p = 1..7$  e $m = 1..3$, com a seguinte semântica:

  > $x_{p,m}$ é verdade  *sse*   o painel que ocupa a posição $p$ é do material $m$

2. De seguida, teremos que modelar cada uma das restrições, acrescentando as fórmulas lógicas correspondentes.

3. Finalmente testamos se o conjunto de restrições é satisfazível e extraimos a solução calculada.  


In [4]:
from z3 import *

posicao = [1,2,3,4,5,6,7]
material = ["madeira", "metal", "vidro"]
x = {}
for p in posicao:
    x[p] = {}
    for m in material:
        x[p][m] = Bool("%d,%s" % (p,m))

s = Solver()

# Cada uma das 7 posições possíveis tem que ter um painel.

for p in posicao:
    s.add(Or([x[p][m] for m in material]))

# Os painéis não podem ser sobrepostos.

for p in posicao:
    for m1 in material:
        for m2 in material:
            if m1 != m2:
                s.add(Or(Not(x[p][m1]), Not(x[p][m2])))

# Os painéis das extremidades não podem ser do mesmo tipo de material.

for m in material:
    s.add(Or(Not(x[1][m]), Not(x[7][m])))
    
# O painel do meio não pode ser de vidro.

s.add(Not(x[4]["vidro"]))

# Todos os materiais têm que ser usados.

for m in material:
    s.add(Or([x[p][m] for p in posicao]))

# Não podem ficar dois painéis de vidro seguidos.

for p in range(1, 7):
    s.add(Or(Not(x[p]["vidro"]), Not(x[p+1]["vidro"])))

# Junto a um painel de metal terá de haver pelo menos um painel de madeira.

for p in posicao:
    if p > 1 and p < 7:
        s.add(Implies(x[p]["metal"], Or(x[p-1]["madeira"], x[p+1]["madeira"])))
    elif p == 1:
        s.add(Implies(x[p]["metal"], x[p+1]["madeira"]))
    elif p == 7:
        s.add(Implies(x[p]["metal"], x[p-1]["madeira"]))

# Se o primeiro painel for de vidro e o segundo de madeira, o último terá que ser de metal.

s.add(Implies(And(x[1]["vidro"], x[2]["madeira"]), x[7]["metal"]))


# Testar se o problema tem solução
if s.check() == sat:
    modelo = s.model()
    print(modelo)    # o modelo encontrado
    print()
    for p in posicao:
        for m in material:
            if is_true(modelo[x[p][m]]):
                print("Na posição %d fica um painel de %s." % (p,m))
else:
  print("Não tem solução.")



[7,madeira = False,
 3,madeira = True,
 3,metal = False,
 3,vidro = False,
 2,madeira = True,
 6,metal = False,
 7,metal = True,
 7,vidro = False,
 4,madeira = True,
 2,vidro = False,
 2,metal = False,
 1,metal = False,
 6,madeira = True,
 1,vidro = True,
 5,vidro = False,
 4,metal = False,
 5,madeira = True,
 1,madeira = False,
 5,metal = False,
 4,vidro = False,
 6,vidro = False]

Na posição 1 fica um painel de vidro.
Na posição 2 fica um painel de madeira.
Na posição 3 fica um painel de madeira.
Na posição 4 fica um painel de madeira.
Na posição 5 fica um painel de madeira.
Na posição 6 fica um painel de madeira.
Na posição 7 fica um painel de metal.


Faça as alterações necessárias ao programa de modo a saber todas as soluções possíveis. Quantas obteve?

In [ ]:
s.push()  # para marcar o ponto em que temos as restrições que correspondem às regras (vamos querer voltar a este ponto)


s.pop()  # retira as restrições que foram acrescentadas até ao último s.push()

Em termos lógicos, como podemos descrever cada uma das seguintes afirmações?
  
 1. *Nas duas primeiras posições terá de haver, pelo menos, um painel de vidro.*
 2. *É possível colocar um painel de madeira na última posição.*
  
  

In [ ]:
s.push ()
s.add(Not(Or(x[1]["vidro"], x[2]["vidro"])))

if s.check() == unsat:
    